# 21 — Student-accessible prediction of contextual teacher features

This notebook inspects the separately frozen `student-accessibility-v1`
experiment. It answers a limited but necessary question: does ordered
skeleton history improve prediction of the current cached target after
observation quality and current posture are represented? It does not train
an S-JEPA encoder or demonstrate distillation.

By default, Run All performs CPU reconstruction refits of the saved selected
models without changing any artifact. Set RECONSTRUCT_MODELS=False in section
2 for strictly no-fit file-integrity inspection. A missing or altered run
causes an error; there is no synthetic replacement for real results. Starting
a new comparison is an explicit CLI operation. A new target or changed search
requires a new run root and prospective protocol.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
from IPython.display import display, SVG, Markdown
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/gavd6_sjepa').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
EVIDENCE = ROOT / 'work/artifacts/iclr-bridge-2026-09-11'
PANEL = ROOT / 'outputs/iclr-bridge-cached-20260911'
def read_json(path):
    return json.loads(Path(path).read_text())
pd.set_option('display.precision', 8)
print('Repository:', ROOT)
print('Execution: read-only evidence inspection + labeled synthetic calculations')

In [ ]:
display(SVG(filename=str(ROOT / 'docs/studies/iclr/figures/02_information_boundary.svg')))

## 1. Confirm the frozen question and lineage

Both panels retain the original 50 windows, 43 sources, five outer folds,
three inner folds, 256-dimensional target and teacher projection. The
teacher saw all 64 frames; target tokens at 38–39 are contextual features.
The first 32 skeleton frames are the only history input. Source separation
controls recordings, not necessarily people appearing in different uploads.

The support reference contains four bins of confidence/validity, endpoint
confidence/validity and frame rate. The posture reference adds valid
frame-31 coordinates. The parent normalization uses a prefix-derived torso
scale, so this is a declared function of history rather than an independently
measured single frame. That qualification matters when interpreting “current.”

In [ ]:
spec = read_json(PANEL / 'config/specification.json')
report = read_json(PANEL / 'reports/panel-report.json')
contract = read_json(PANEL / 'config/run-contract.json')
display(pd.DataFrame([{'version':spec['version'], 'clips':report['clips'],
                      'sources':report['sources'], 'seed_policy':spec['seed_policy'],
                      'teacher_evidence':contract['teacher_evidence']}]))
display(pd.DataFrame({'panel':list(spec['baseline_dimensions']),
                      'reference_features':list(spec['baseline_dimensions'].values())}))
print('Primary:', spec['primary_contrast'])
print('Created UTC:', contract['created_utc'])

## 2. Reconstruct before interpreting

The verifier checks source/cache snapshots, feature schemas, train-only
preprocessing, controls and typed models. It refits the selected linear
solutions, reconstructs each held-out prediction in standardized and raw
units, and recomputes scores and all paired bootstrap draws. Candidate
records are checked for pooling and selection; all rejected candidates
are not independently refitted by this command. That is its stated boundary.

Verification is read-only. It requires the compatible frozen implementation
and intact parent artifacts. Passing the following cell is stronger than
seeing a checkpoint count or a correct checksum.

In [ ]:
from gavd6_sjepa.research_directions.iclr_bridge.verification_supplement import verify_panel_supplement
from gavd6_sjepa.research_directions.iclr_bridge.inspection import inspect_cached_panel
RECONSTRUCT_MODELS = True  # False checks lineage/digests only, with strictly no fitting.
verification = verify_panel_supplement(PANEL) if RECONSTRUCT_MODELS else inspect_cached_panel(PANEL)
display(pd.DataFrame([verification]))
if RECONSTRUCT_MODELS:
    assert verification['status'] == 'passed'
    assert verification['original_verification']['parent_artifacts_unchanged']
else:
    assert verification['status'] == 'integrity_checked'
    print('No numerical reconstruction performed in this mode.')

## 3. Understand what was fitted and selected

Each reference has six ridge penalties. Each arm has 36 pairs of positive
penalties for the reference and history blocks, plus one exact baseline.
Training weights give each source equal total influence. The intercept is
unpenalized. Inner validation chooses a model; the outer sources evaluate
that choice. A nonzero winner can still perform worse on outer sources.

No-skeleton retains time-varying validity. Shuffle moves coordinates,
confidence and validity together in four-frame blocks. Mismatch uses a
different-source donor from the same partition and never uses targets.
All arms retain the recipient's reference inputs. Supported feature counts
can differ after controls, so nominal parameter counts are only one part
of the comparison.

The real history block also supplies validity-conditioned confidence means;
the reference uses raw confidence means. These are different features under
missingness. Real minus no-skeleton therefore tests a coordinate/confidence
bundle, not isolated motion. A subsequent confidence/validity-only history
arm would control that route more tightly. In two posture folds, real also
selected a different reference-block penalty. The shared-baseline gain
includes this regularization change.

In [ ]:
selections = pd.read_csv(PANEL / 'reports/selections.csv')
display(pd.crosstab([selections.panel, selections.arm], selections.type))
candidate_rows, diagnostic_rows = [], []
for panel in spec['panels']:
    for fold in range(5):
        folder=PANEL / f'models/{panel}/fold-{fold}'
        for arm, record in read_json(folder/'selection.json').items():
            for candidate in record['candidates']:
                candidate_rows.append({'panel':panel,'fold':fold,'arm':arm,
                    **{k:candidate[k] for k in ('candidate_id','pooled_loss','improvement_over_baseline','valid','selected','selection_reason')}})
        diagnostic_rows.extend({'panel':panel,'fold':fold,**r} for r in read_json(folder/'diagnostics.json'))
candidates=pd.DataFrame(candidate_rows)
diagnostics=pd.DataFrame(diagnostic_rows)
print('Pooled candidates:',len(candidates),'valid:',int(candidates.valid.sum()))
display(candidates[candidates.selected])
display(diagnostics[['panel','fold','arm','x_features','x_supported','s_features','s_supported','training_mse','selected_inner_mse']])

## 4. Read the matched effect and its uncertainty

All scores use source-balanced featurewise predictive R² relative to the
outer-training mean. The declared primary contrast is posture-panel real
minus no-skeleton. Other contrasts help explain the result but cannot
replace the primary after inspection. Displaying eight decimal places
preserves the sign and scale of small effects.

The 2,000 bootstrap draws resample whole sources with replacement and keep
repeated-source multiplicities. They reuse fitted predictions; they do not
repeat fitting or model selection. Thus these are conditional development
intervals, not independent replication. This deterministic predictor has
one result, even though the artifact identity uses seed 0.

In [ ]:
score_rows, contrast_rows = [], []
for panel,p in report['panels'].items():
    score_rows += [{'panel':panel,'arm':arm,**scores} for arm,scores in p['scores'].items()]
    contrast_rows += [{'panel':panel,'contrast':name,**values} for name,values in p['contrasts'].items()]
display(pd.DataFrame(score_rows))
display(pd.DataFrame(contrast_rows))
print('Measurement complete:',report['measurement_complete'])
print('Frozen descriptive status:',report['status'])
print('Scientific advance:',report['scientific_advance'])

## 5. Decide what follows, without changing the rule

The frozen descriptive `development_lead` requires the primary 95% interval
to exclude zero positively and real to beat shuffle and mismatch in point
estimates. Otherwise a complete run reports `no_supported_temporal_lead`.
An invalid required comparison reports `incomplete_measurement`. Neither
descriptive outcome changes the earlier direct-v3 STOP, and neither can
authorize a paper claiming successful student transfer.

If there is a lead, freeze a future-only, physical-time target and one
student-training comparison before collecting confirmation data. If there
is no lead, first examine target meaning and direct pose prediction, then
choose a separately specified representation or target study. Repeatedly
replacing fifty videos until an effect appears would confound model
development with confirmation.

## 6. Explicit reproduction and recovery

The following commands are shown as text. Running this notebook does not
execute them. `run` resumes verified completed folds and verifies a sealed
completed stage. `freeze` requires a new directory. A protocol or code
amendment belongs in a separate experiment.

```bash
.venv/bin/python scripts/research_directions/iclr_bridge/run_cached_panel.py run --output-root outputs/iclr-bridge-cached-20260911
.venv/bin/python scripts/research_directions/iclr_bridge/run_cached_panel.py verify --output-root outputs/iclr-bridge-cached-20260911
```

The [frozen protocol](docs/studies/iclr/02_cached_panel_protocol.md) specifies
the construction and the [validation report](docs/studies/iclr/03_implementation_and_validation.md)
records what actually ran. Continue with notebook **22** to design the
next target and student study.